In [37]:
import yatzy.graph.build_graph as bg

In [38]:
all_dice_sets = bg.generate_all_dice_sets()
all_start_states = bg.reduce_start_states(bg.generate_start_states())
all_reroll_masks = bg.generate_all_reroll_masks(num_dices=5)
all_reroll_probs = bg.compute_all_reroll_probabilities(all_dice_sets, all_reroll_masks)
all_dice_set_probabilities = bg.generate_all_dice_set_probabilities(all_dice_sets)
all_state_integers = bg.generate_all_state_integers()
all_reroll_masks_by_filter = bg.generate_reroll_masks(all_dice_sets)
all_possible_reroll_masks = bg.generate_all_possible_reroll_masks(all_dice_sets, all_reroll_masks_by_filter)

Loading probabilities from file: /Users/langkilde/IdeaProjects/yatzy/data/reroll_probabilities.pkl.gz


In [39]:
spark = bg.setup_spark()
# print(json.dumps(spark.sparkContext.getConf().getAll(), indent=4))
spark_context = spark.sparkContext
number_of_partitions = 200

In [40]:
expected_state_scores = {}
exit_states = bg.filter_states_by_depth(all_start_states, 0)
exit_states = spark_context.parallelize(exit_states, numSlices=number_of_partitions)
expected_state_scores = bg.set_exit_state_values(exit_states, expected_state_scores)

In [41]:
dice_transitions_rdd = spark_context.parallelize(all_dice_sets, numSlices=number_of_partitions) \
        .map(lambda dice_set: (dice_set, bg.get_reroll_filter(dice_set))) \
        .flatMap(lambda row: bg.expand_by_reroll_masks(row[0], row[1], all_reroll_masks_by_filter)) \
        .flatMap(lambda row: bg.expand_by_target_dice_set(row[0], row[1], all_dice_sets)) \
        .map(lambda row: (row[0], row[1], row[2], bg.get_reroll_prob_from_map(all_reroll_probs, row[0], row[1], row[2])))\
.filter(lambda row: row[3] > 0).repartition(number_of_partitions)

In [44]:
states_with_same_depth = bg.filter_states_by_depth(all_start_states, 1)
len(states_with_same_depth)

819

In [45]:
states_rdd = spark_context.parallelize(states_with_same_depth, numSlices=number_of_partitions)

In [47]:
enriched_states_rdd = states_rdd.map(lambda state: (state, bg.decode_state_integer(state))) \
                .map(lambda row: (row[0], row[1][0], row[1][1])) \
                .flatMap(lambda row: bg.expand_by_all_dice_sets(row[0], row[1], row[2], all_dice_sets)) \
                .map(lambda row:
                     (row[0], row[1], row[2], row[3], bg.get_max_exit_score(row[0], row[1], row[2], row[3], False))) \
                .map(lambda row: (row[0], row[1], row[2], row[3], row[4][0], row[4][1], row[4][2])) \
                .map(lambda row:
                     (row[0], row[1], row[2], row[3], row[4], row[5], row[6], bg.encode_state_integer(row[4], row[6]))) \
                .map(lambda row:
                     (row[0], row[1], row[2], row[3], row[4], row[5], row[6], row[7], expected_state_scores[row[7]])) \
                .map(lambda row:
                     (row[0], row[1], row[2], row[3], row[4], row[5], row[6], row[7], row[8], row[5] + row[8]))

In [51]:
from pyspark.sql.functions import col, concat_ws, udf, row_number, desc

enriched_states_df = (enriched_states_rdd \
                                  .toDF(['state',
                                         'upper_score',
                                         'scored_categories',
                                         'exit_dice_set',
                                         'new_upper_score',
                                         'max_additional_score',
                                         'new_scored_categories',
                                         'next_state',
                                         'next_state_potential',
                                         'exit_dice_set_value']) \
                                  .withColumn('exit_dice_set', concat_ws('', 'exit_dice_set'))
                                  .repartition(number_of_partitions))

In [88]:
enriched_states_df.take(1)

[Row(state=15871, upper_score=1, scored_categories='1110111111111', exit_dice_set='13666', new_upper_score=1, max_additional_score=0, new_scored_categories='1111111111111', next_state=16383, next_state_potential=0, exit_dice_set_value=0)]

In [54]:
dice_transitions_df = dice_transitions_rdd.toDF(['r_start', 'r_prim', 'r_target', 'prob']) \
                .withColumn('r_start', concat_ws('', 'r_start')) \
                .withColumn('r_prim', concat_ws('', 'r_prim')) \
                .withColumn('r_target', concat_ws('', 'r_target'))

In [56]:
joined_df = dice_transitions_df \
                .join(enriched_states_df, col('exit_dice_set') == col('r_target'), 'left') \
                .repartition(number_of_partitions)

In [87]:
joined_df.take(1)

[Row(state=147391, r_start='11115', r_prim='11110', sum(target_dice_set_potential)=6.707079487852752)]

In [59]:
second_reroll_df = joined_df.repartition(number_of_partitions) \
                .withColumn('target_dice_set_potential', (col('prob') * col('exit_dice_set_value')).cast('float')) \
                .select('state', 'r_start', 'r_prim', 'target_dice_set_potential') \
                .groupBy("state", "r_start", "r_prim") \
                .sum("target_dice_set_potential")
second_reroll_df = second_reroll_df.repartition(number_of_partitions)

In [84]:
second_reroll_df.take(1)

[Row(state=102399, r_start='12256', r_prim='00100', sum(target_dice_set_potential)=1.1666667014360428)]

In [63]:
from pyspark.sql import Window
windowSpec = Window.partitionBy("state", "r_start") \
                .orderBy(desc("sum(target_dice_set_potential)"))
df_with_rownum = second_reroll_df.repartition(number_of_partitions) \
    .withColumn("row_num", row_number().over(windowSpec))
second_reroll_dice_set_values_df = df_with_rownum.filter(df_with_rownum["row_num"] == 1).drop("row_num")
second_reroll_dice_set_values_df = second_reroll_dice_set_values_df \
    .select('state', 'r_start', 'sum(target_dice_set_potential)') \
    .withColumnRenamed("r_start", "dice_set") \
    .withColumnRenamed("sum(target_dice_set_potential)", "dice_set_value")
second_reroll_dice_set_values_df = second_reroll_dice_set_values_df.repartition(number_of_partitions)

In [82]:
second_reroll_dice_set_values_df.take(1)

[Row(state=299007, dice_set='23456', dice_set_value=0.8333333419868723)]

In [66]:
joined_df = dice_transitions_df \
                .join(second_reroll_dice_set_values_df,
                      dice_transitions_df['r_target'] == second_reroll_dice_set_values_df['dice_set'],
                      how='inner') \
                .withColumn('target_dice_set_potential', (col('prob') * col('dice_set_value')).cast('float')) \
                .groupBy("state", "r_start", "r_prim") \
                .sum("target_dice_set_potential") \
                .alias("reroll_mask_potential")

In [81]:
joined_df.take(1)

[Row(state=147391, r_start='11115', r_prim='11110', sum(target_dice_set_potential)=6.707079487852752)]

In [ ]:
windowSpec = Window.partitionBy("state", "r_start") \
                .orderBy(desc("sum(target_dice_set_potential)"))
df_with_rownum = joined_df.repartition(number_of_partitions) \
    .withColumn("row_num", row_number().over(windowSpec))
joined_df = df_with_rownum.filter(df_with_rownum["row_num"] == 1).drop("row_num")

In [71]:
final_df = joined_df.rdd.map(lambda row: (row['state'],
              bg.decode_state_integer(row['state']),
              row['r_start'],
              row['sum(target_dice_set_potential)'] *
              all_dice_set_probabilities["".join(map(str, row['r_start']))])) \
        .toDF(["state", "upper+cat", "dice_set", "value"])

In [79]:
final_df.take(1)

[Row(state=147391, upper+cat=Row(_1=17, _2='1111110111111'), dice_set='11115', value=0.004312679711839475)]

In [72]:
state_values_df = final_df.groupBy("state").sum("value").sort("sum(value)", ascending=False)
state_values_list = state_values_df.rdd.collect()

In [80]:
state_values_df.take(1)

[Row(state=524285, sum(value)=1262.3406674558119)]

In [76]:
state_values_list

[Row(state=524285, sum(value)=1262.3406674558119),
 Row(state=524159, sum(value)=989.2845124137589),
 Row(state=524031, sum(value)=957.3000936183225),
 Row(state=523775, sum(value)=923.217191849678),
 Row(state=524223, sum(value)=913.3600648268285),
 Row(state=523263, sum(value)=889.9347462744844),
 Row(state=522239, sum(value)=858.2826701201436),
 Row(state=507775, sum(value)=849.7369710730003),
 Row(state=515967, sum(value)=849.7369710730003),
 Row(state=491391, sum(value)=849.7369710730002),
 Row(state=475007, sum(value)=849.7369710730002),
 Row(state=499583, sum(value)=849.7369710730002),
 Row(state=483199, sum(value)=849.7369710730001),
 Row(state=524283, sum(value)=843.7503313785095),
 Row(state=524279, sum(value)=829.5174128535964),
 Row(state=520191, sum(value)=827.0918473898141),
 Row(state=524271, sum(value)=822.7008127254235),
 Row(state=491263, sum(value)=818.4422420908327),
 Row(state=515839, sum(value)=818.4422420908327),
 Row(state=499455, sum(value)=818.4422420908326),


In [92]:
bg.decode_state_integer(57342)

(6, '1111111111110')